In [16]:
 # ============================================================
#    DATA CLEANING

# ------------------------------------------------------------
# 1. IMPORT LIBRARIES
# ------------------------------------------------------------

import pandas as pd
import numpy as np

# Display all columns
pd.set_option("display.max_columns", None)

# ------------------------------------------------------------
# 2. LOAD DATASET
# ------------------------------------------------------------

file_path = "ecommerce_customer_data_custom_ratios.csv"

data = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print("Original Shape:", data.shape)


# ------------------------------------------------------------
# 3. STANDARDIZE COLUMN NAMES
# ------------------------------------------------------------

# Remove extra spaces, convert to lowercase,
# and replace spaces with underscores

data.columns = (
    data.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

print("\nColumns:")
print(data.columns.tolist())


# ------------------------------------------------------------
# 4. CREATE DATA COPY
# ------------------------------------------------------------

# Keep the original data unchanged

df = data.copy()

print("\nWorking dataset shape:", df.shape)


# ------------------------------------------------------------
# 5. CONVERT PURCHASE DATE
# ------------------------------------------------------------

# Convert Purchase Date into datetime format

df["purchase_date"] = pd.to_datetime(
    df["purchase_date"],
    errors="coerce"
)

# Check invalid dates

print(
    "\nInvalid dates:",
    df["purchase_date"].isna().sum()
)

# Display date range

print(
    "Date Range:",
    df["purchase_date"].min(),
    "to",
    df["purchase_date"].max()
)


# ------------------------------------------------------------
# 6. CONVERT NUMERICAL COLUMNS
# ------------------------------------------------------------

numeric_columns = [
    "product_price",
    "quantity",
    "total_purchase_amount",
    "customer_age",
    "returns",
    "churn"
]

# Convert columns to numeric
# Invalid values become NaN

for column in numeric_columns:
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )

print("\nNumerical columns converted.")


# ------------------------------------------------------------
# 7. CHECK MISSING VALUES
# ------------------------------------------------------------

missing_values = pd.DataFrame({
    "Missing_Count": df.isnull().sum(),
    "Missing_Percentage": (
        df.isnull().sum() / len(df) * 100
    ).round(2)
})

print("\nMissing Value Report:")
print(
    missing_values.sort_values(
        "Missing_Percentage",
        ascending=False
    )
)


# ------------------------------------------------------------
# 8. HANDLE MISSING RETURNS
# ------------------------------------------------------------

# Check return values before treatment

print("\nReturns before treatment:")
print(df["returns"].value_counts(dropna=False))

# Treat missing return records as no recorded return

df["returns"] = df["returns"].fillna(0)

print(
    "\nMissing Returns after treatment:",
    df["returns"].isna().sum()
)


# ------------------------------------------------------------
# 9. HANDLE MISSING CUSTOMER AGE
# ------------------------------------------------------------

# Replace missing ages with the median age

if df["customer_age"].isna().sum() > 0:

    median_age = df["customer_age"].median()

    df["customer_age"] = df["customer_age"].fillna(
        median_age
    )

print(
    "\nMissing Customer Age:",
    df["customer_age"].isna().sum()
)


# ------------------------------------------------------------
# 10. HANDLE MISSING CHURN
# ------------------------------------------------------------

# Check churn values

print("\nChurn values:")
print(df["churn"].value_counts(dropna=False))

# If churn contains missing values,
# remove those records because churn is an important
# target variable and should not be guessed

df = df.dropna(subset=["churn"])


# ------------------------------------------------------------
# 11. REMOVE RECORDS WITH INVALID DATES
# ------------------------------------------------------------

# Purchase date is essential for recency calculation

df = df.dropna(
    subset=["purchase_date"]
)

print(
    "\nShape after date cleaning:",
    df.shape
)


# ------------------------------------------------------------
# 12. CHECK DUPLICATE ROWS
# ------------------------------------------------------------

duplicate_count = df.duplicated().sum()

print(
    "\nDuplicate rows:",
    duplicate_count
)

# Remove exact duplicate rows

df = df.drop_duplicates()

print(
    "Shape after duplicate removal:",
    df.shape
)


# ------------------------------------------------------------
# 13. CHECK INVALID QUANTITY
# ------------------------------------------------------------

print(
    "\nInvalid Quantity:",
    (df["quantity"] <= 0).sum()
)

# Remove transactions with zero/negative quantity

df = df[df["quantity"] > 0]


# ------------------------------------------------------------
# 14. CHECK INVALID PRODUCT PRICE
# ------------------------------------------------------------

print(
    "\nInvalid Product Price:",
    (df["product_price"] <= 0).sum()
)

# Remove transactions with zero/negative price

df = df[df["product_price"] > 0]


# ------------------------------------------------------------
# 15. CHECK INVALID PURCHASE AMOUNT
# ------------------------------------------------------------

print(
    "\nInvalid Purchase Amount:",
    (df["total_purchase_amount"] <= 0).sum()
)

# Remove invalid purchase amounts

df = df[
    df["total_purchase_amount"] > 0
]


# ------------------------------------------------------------
# 16. CHECK CUSTOMER AGE RANGE
# ------------------------------------------------------------

invalid_age = (
    (df["customer_age"] < 18) |
    (df["customer_age"] > 100)
)

print(
    "\nInvalid age records:",
    invalid_age.sum()
)

# Remove impossible age values

df = df[
    ~invalid_age
]


# ------------------------------------------------------------
# 17. CHECK AGE COLUMN
# ------------------------------------------------------------

# The dataset contains both customer_age and age.
# Check whether they contain the same information.

if "age" in df.columns:

    age_difference = (
        df["customer_age"] != df["age"]
    ).sum()

    print(
        "\nAge mismatches:",
        age_difference
    )

    # Remove duplicate age column

    df = df.drop(
        columns=["age"]
    )


# ------------------------------------------------------------
# 18. CLEAN CATEGORICAL COLUMNS
# ------------------------------------------------------------

categorical_columns = [
    "product_category",
    "payment_method",
    "gender"
]

for column in categorical_columns:

    # Remove extra spaces

    df[column] = (
        df[column]
        .astype(str)
        .str.strip()
    )


# ------------------------------------------------------------
# 19. CHECK CUSTOMER IDs
# ------------------------------------------------------------

print(
    "\nMissing Customer IDs:",
    df["customer_id"].isna().sum()
)

# Remove transactions without customer ID

df = df.dropna(
    subset=["customer_id"]
)


# ------------------------------------------------------------
# 20. CHECK DATASET AFTER CLEANING
# ------------------------------------------------------------

print("\nFinal Cleaned Transaction Shape:")
print(df.shape)

print("\nRemaining Missing Values:")
print(df.isnull().sum())


# ------------------------------------------------------------
# 21. CREATE DATE FEATURES
# ------------------------------------------------------------

df["year"] = df["purchase_date"].dt.year

df["month"] = df["purchase_date"].dt.month

df["month_name"] = (
    df["purchase_date"].dt.month_name()
)

df["quarter"] = (
    df["purchase_date"].dt.quarter
)

df["day_of_week"] = (
    df["purchase_date"].dt.day_name()
)


# ------------------------------------------------------------
# 22. CREATE CALCULATED PURCHASE AMOUNT
# ------------------------------------------------------------

# Calculate expected transaction amount

df["calculated_amount"] = (
    df["product_price"] *
    df["quantity"]
)

# Calculate difference from recorded amount

df["amount_difference"] = (
    df["total_purchase_amount"] -
    df["calculated_amount"]
)

print("\nPurchase Amount Validation:")
print(
    df["amount_difference"].describe()
)


# ------------------------------------------------------------
# 23. REMOVE TEMPORARY VALIDATION COLUMNS
# ------------------------------------------------------------

df = df.drop(
    columns=[
        "calculated_amount",
        "amount_difference"
    ]
)

# ------------------------------------------------------------
# 24. CREATE CUSTOMER-LEVEL DATASET
# ------------------------------------------------------------

customer_df = (
    df.groupby(
        ["customer_id", "customer_name"]
    )
    .agg(
        total_spend=(
            "total_purchase_amount",
            "sum"
        ),

        purchase_frequency=(
            "customer_id",
            "count"
        ),

        total_quantity=(
            "quantity",
            "sum"
        ),

        average_order_value=(
            "total_purchase_amount",
            "mean"
        ),

        average_product_price=(
            "product_price",
            "mean"
        ),

        first_purchase_date=(
            "purchase_date",
            "min"
        ),

        last_purchase_date=(
            "purchase_date",
            "max"
        ),

        total_returns=(
            "returns",
            "sum"
        ),

        churn=(
            "churn",
            "max"
        )
    )
    .reset_index()
)


# ------------------------------------------------------------
# 25. CREATE RECENCY
# ------------------------------------------------------------

# Use the latest transaction date as the analysis date

analysis_date = df["purchase_date"].max()

print(
    "\nAnalysis Date:",
    analysis_date
)

# Calculate days since last purchase

customer_df["recency"] = (
    analysis_date -
    customer_df["last_purchase_date"]
).dt.days


# ------------------------------------------------------------
# 26. CREATE CUSTOMER TENURE
# ------------------------------------------------------------

# Number of days between first and last purchase

customer_df["customer_tenure_days"] = (
    customer_df["last_purchase_date"] -
    customer_df["first_purchase_date"]
).dt.days

# Convert tenure into months

customer_df["customer_tenure_months"] = (
    customer_df["customer_tenure_days"] / 30
).round(1)


# ------------------------------------------------------------
# 27. CREATE RETURN RATE
# ------------------------------------------------------------

customer_df["return_rate"] = (
    customer_df["total_returns"] /
    customer_df["purchase_frequency"]
)

# Convert to percentage

customer_df["return_rate_pct"] = (
    customer_df["return_rate"] * 100
).round(2)


# ------------------------------------------------------------
# 28. CUSTOMER AGE AND GENDER
# ------------------------------------------------------------

customer_info = (
    df.groupby("customer_id")
    .agg(
        customer_age=(
            "customer_age",
            "first"
        ),

        gender=(
            "gender",
            "first"
        )
    )
    .reset_index()
)

customer_df = customer_df.merge(
    customer_info,
    on="customer_id",
    how="left"
)


# ------------------------------------------------------------
# 29. PREFERRED PRODUCT CATEGORY
# ------------------------------------------------------------

# Find the most frequently purchased category
# for each customer

category_preference = (
    df.groupby("customer_id")["product_category"]
    .agg(
        lambda x:
        x.mode().iloc[0]
        if not x.mode().empty
        else "Unknown"
    )
    .reset_index()
)

category_preference.columns = [
    "customer_id",
    "preferred_category"
]

customer_df = customer_df.merge(
    category_preference,
    on="customer_id",
    how="left"
)


# ------------------------------------------------------------
# 30. PREFERRED PAYMENT METHOD
# ------------------------------------------------------------

# Find the most frequently used payment method

payment_preference = (
    df.groupby("customer_id")["payment_method"]
    .agg(
        lambda x:
        x.mode().iloc[0]
        if not x.mode().empty
        else "Unknown"
    )
    .reset_index()
)

payment_preference.columns = [
    "customer_id",
    "preferred_payment_method"
]

customer_df = customer_df.merge(
    payment_preference,
    on="customer_id",
    how="left"
)


# ------------------------------------------------------------
# 31. VERIFY CUSTOMER-LEVEL DATA
# ------------------------------------------------------------

print("\nCustomer-Level Dataset Shape:")
print(customer_df.shape)

print("\nCustomer-Level Columns:")
print(customer_df.columns.tolist())

print("\nFirst 5 Customers:")
print(customer_df.head())


# ------------------------------------------------------------
# 32. CHECK DUPLICATE CUSTOMERS
# ------------------------------------------------------------

duplicate_customers = (
    customer_df["customer_id"]
    .duplicated()
    .sum()
)

print(
    "\nDuplicate Customers:",
    duplicate_customers
)


# ------------------------------------------------------------
# 33. CUSTOMER STATISTICAL SUMMARY
# ------------------------------------------------------------

customer_numeric_columns = [
    "total_spend",
    "purchase_frequency",
    "total_quantity",
    "average_order_value",
    "average_product_price",
    "recency",
    "customer_tenure_days",
    "total_returns",
    "return_rate_pct"
]

print("\nCustomer Statistics:")

print(
    customer_df[
        customer_numeric_columns
    ].describe()
)


# ------------------------------------------------------------
# 34. OUTLIER DETECTION
# ------------------------------------------------------------

# Detect potential outliers using IQR.
# Valid high-value customers will NOT be removed.

for column in [
    "total_spend",
    "purchase_frequency",
    "total_quantity",
    "average_order_value",
    "recency"
]:

    Q1 = customer_df[column].quantile(0.25)

    Q3 = customer_df[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_limit = Q1 - 1.5 * IQR

    upper_limit = Q3 + 1.5 * IQR

    outliers = (
        (customer_df[column] < lower_limit) |
        (customer_df[column] > upper_limit)
    ).sum()

    print(
        f"{column}: {outliers} potential outliers"
    )


# ------------------------------------------------------------
# 35. CREATE RFM BASE DATASET
# ------------------------------------------------------------

rfm = customer_df[
    [
        "customer_id",
        "customer_name",
        "recency",
        "purchase_frequency",
        "total_spend",
        "churn"
    ]
].copy()


# Rename columns according to RFM terminology

rfm = rfm.rename(
    columns={
        "purchase_frequency": "frequency",
        "total_spend": "monetary"
    }
)


# ------------------------------------------------------------
# 36. CHECK RFM DATA
# ------------------------------------------------------------

print("\nRFM Dataset Shape:")
print(rfm.shape)

print("\nRFM Dataset:")
print(rfm.head())

print("\nRFM Missing Values:")
print(rfm.isnull().sum())


# ============================================================
# SAVE OUTPUT FILES
# ============================================================


# ------------------------------------------------------------
# 37. SAVE CLEANED TRANSACTION DATA
# ------------------------------------------------------------

df.to_csv(
    "ecommerce_cleaned_transactions.csv",
    index=False
)

print(
    "\nCleaned transaction dataset saved!"
)


# ------------------------------------------------------------
# 38. SAVE CUSTOMER-LEVEL FEATURES
# ------------------------------------------------------------

customer_df.to_csv(
    "customer_level_features.csv",
    index=False
)

print(
    "Customer-level feature dataset saved!"
)


# ------------------------------------------------------------
# 39. SAVE RFM BASE DATASET
# ------------------------------------------------------------

rfm.to_csv(
    "customer_rfm_base.csv",
    index=False
)

print(
    "RFM base dataset saved!"
)


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("PHASE 2 COMPLETED")
print("=" * 60)

print(
    "Cleaned Transactions:",
    len(df)
)

print(
    "Unique Customers:",
    customer_df["customer_id"].nunique()
)

print(
    "RFM Customers:",
    len(rfm)
)

print(
    "Analysis Date:",
    analysis_date
)

print("\nOutput Files:")
print("1. ecommerce_cleaned_transactions.csv")
print("2. customer_level_features.csv")
print("3. customer_rfm_base.csv")

Dataset loaded successfully!
Original Shape: (250000, 13)

Columns:
['customer_id', 'purchase_date', 'product_category', 'product_price', 'quantity', 'total_purchase_amount', 'payment_method', 'customer_age', 'returns', 'customer_name', 'age', 'gender', 'churn']

Working dataset shape: (250000, 13)

Invalid dates: 0
Date Range: 2020-01-01 00:15:00 to 2023-09-15 12:24:08

Numerical columns converted.

Missing Value Report:
                       Missing_Count  Missing_Percentage
returns                        47596               19.04
customer_id                        0                0.00
purchase_date                      0                0.00
product_category                   0                0.00
product_price                      0                0.00
quantity                           0                0.00
total_purchase_amount              0                0.00
payment_method                     0                0.00
customer_age                       0                0.00
cust